<div class="alert alert-warning">

# PS 8 — Neural Networks and Backpropagation

In this problem set you will build a **two-layer feedforward neural network** from scratch and train it with **backpropagation**.

## Motivating example: social awkwardness

Imagine you arrive at a party. You feel awkward when **exactly one** person is dressed up:

| You | Others | Awkward? |
|:---:|:---:|:---:|
| casual | casual | no |
| dressed up | casual | **yes** |
| casual | dressed up | **yes** |
| dressed up | dressed up | no |

This is the **XOR pattern**: the outcome is 1 when the two inputs *differ*, and 0 when they *match*. Notice that no single threshold on either feature can solve it — knowing only whether *you* are dressed up tells you nothing without knowing what *others* are wearing.

## Why this matters for AI

In 1969, Minsky & Papert proved in their book *Perceptrons* that a single-layer network **cannot solve XOR** — a finding that contributed to the first "AI winter." In 1986, Rumelhart, Hinton & Williams showed that adding a hidden layer and training it with backpropagation solves the problem, reigniting the field.

We will:
1. Generate a **noisy XOR** dataset and visualize it
2. Train a **single-layer** network and confirm it fails
3. Understand **why** it fails geometrically
4. Build a **two-layer** network and derive the backpropagation equations
5. Train it and show it succeeds
6. Visualize **what the hidden units learned**

</div>

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

---

# 1. The Dataset

We model our party scenario with two continuous features:

- $x_1$ = how formally **you** are dressed (0 = very casual, 1 = very dressed up)
- $x_2$ = how formally **others** are dressed (0 = very casual, 1 = very dressed up)

The label follows the XOR rule — **class 1 (awkward)** when the two values are on opposite sides; **class 0 (fine)** when they match:

| $x_1$ (you) | $x_2$ (others) | label |
|:---:|:---:|:---:|
| 0 (casual) | 0 (casual) | 0 — fine |
| 0 (casual) | 1 (dressed up) | 1 — awkward |
| 1 (dressed up) | 0 (casual) | 1 — awkward |
| 1 (dressed up) | 1 (dressed up) | 0 — fine |

We use a **noisy** version: instead of four exact points, we sample clusters of people around each of the four corners of the unit square, adding Gaussian noise to represent natural variation in dress level.

<div class="alert alert-success">

## Exercise 1

**A.** We provided the code for `make_noisy_xor` to generate the dataset. Complete the cell below plot it, coloring points by class.

**B.** Can you draw a single straight line to separate the two classes (awkward vs. fine)? Why?

</div>

In [ ]:
def make_noisy_xor(n=200, noise=0.2, seed=42):
    """
    Generate noisy XOR data.
    Class 0 (fine):    clusters near (0,0) and (1,1)
    Class 1 (awkward): clusters near (0,1) and (1,0)

    Returns
    -------
    X : ndarray, shape (n, 2)
    y : ndarray, shape (n,)  — labels 0 or 1
    """
    rng = np.random.default_rng(seed)
    n_per = n // 4
    # (center_x, center_y, label)
    corners = [(0, 0, 0), (1, 1, 0), (0, 1, 1), (1, 0, 1)]
    X_parts, y_parts = [], []
    for cx, cy, label in corners:
        X_parts.append(rng.normal([cx, cy], noise, size=(n_per, 2)))
        y_parts.append(np.full(n_per, label))
    return np.vstack(X_parts), np.concatenate(y_parts)

# X: ndarray of shape (n, 2). Each row X[i] = [x1, x2] is one feature (you, others).
# y: ndarray of shape (n,). Each label y[i] is the class (0 or 1) for the same example X[i].
X, y = make_noisy_xor()

# Exercise 1A: Plot the data
fig, ax = plt.subplots(figsize=(5, 5))
# YOUR CODE HERE -- HINT: use ax.scatter
ax.set_xlabel('$x_1$ — your formality')
ax.set_ylabel('$x_2$ — others\' formality')
ax.set_title('Social congruence (noisy XOR)')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">
    Exercise 1B: Can you draw a single straight line to separate the two classes (awkward vs. fine)? Why?
    Write down your answer below
</div>



---

# 2. A Single-Layer Network
![One layer visual](one_layer.png)

A single-layer network takes the two input features $x_1$ and $x_2$ and makes a prediction in two steps:

**Step 1 — Weighted sum:**
$$z = w_1 \cdot x_1 + w_2 \cdot x_2 + b$$
where $w_1, w_2$ are learned weights and $b$ is a bias term.

**Step 2 — Sigmoid activation:**
$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$
The sigmoid function squashes $z$ into a number between 0 and 1, which we interpret as the probability of class 1.



## Measuring error: Binary cross-entropy loss

In class, we measured the error signal as the error distance. A better way to measure error in categorization tasks is called the Binary cross-entropy loss - this measure has better properties when your output is like a probability: it's like a log-likelihood for model fitting.

We measure how wrong the predictions are with the binary cross-entropy loss, averaged over all $n$ training examples, with this formula:
$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n} \bigl[y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\bigr]$$
This is large when the predicted probability is far from the true label, and small when predictions are confident and correct.

## Updating the weights: Gradient descent

After a forward pass through all the data, define the **error** for each example $i$:
$$\delta_i = \hat{y}_i - y_i$$
(positive when we predicted too high, negative when we predicted too low). Then update each weight in proportion to how much its input contributed to the error:
$$w_1 \leftarrow w_1 - \frac{\eta}{n}\sum_{i=1}^n \delta_i \cdot x_{i,1}$$
$$w_2 \leftarrow w_2 - \frac{\eta}{n}\sum_{i=1}^n \delta_i \cdot x_{i,2}$$
$$b \leftarrow b - \frac{\eta}{n}\sum_{i=1}^n \delta_i$$
Here $\eta$ (eta) is the **learning rate**, which controls the step size.

---

**NumPy note: the `@` operator**

Throughout this notebook you will see expressions like `X @ w`. The `@` symbol is NumPy's matrix-multiply operator. With `X` of shape `(n, 2)` and `w` of shape `(2,)`, it computes the weighted sum $w_1 x_{i,1} + w_2 x_{i,2}$ for **every example $i$ at once**, returning a vector of shape `(n,)`. It is equivalent to a loop over examples, but written as a single operation:

```python
# These are identical:
z = X @ w + b                        # matrix form — one line, no loop
z = np.array([w @ X[i] + b          # loop form
              for i in range(n)])
```

You will use the same idea with matrices: `X @ W1.T` applies multiple weight vectors simultaneously, and `X.T @ delta` sums a product over all examples at once.

---

<div class="alert alert-success">

## Exercise 2 (A–B)

**A.** Implement `sigmoid(z)` and `bce_loss(y, y_hat)`, according to the formulas above.

**B.** Implement `forward_1layer(X, w, b)` which computes $z = w_1 x_1 + w_2 x_2 + b$ for all examples at once and returns the vector of predictions $\hat{y}$ (shape `(n,)`).

</div>

In [ ]:
def sigmoid(z):
    """Element-wise sigmoid function."""
    # YOUR CODE HERE (REPLACE)
    return None


def bce_loss(y, y_hat, eps=1e-8):
    """Mean binary cross-entropy loss."""
    # YOUR CODE HERE (REPLACE)
    return None


def forward_1layer(X, w, b):
    """
    Single-layer forward pass.

    Parameters
    ----------
    X : (n, 2)
    w : (2,)
    b : float

    Returns
    -------
    y_hat : (n,)
    """
    # YOUR CODE HERE (REPLACE)
    return None


In [ ]:
"""Check sigmoid, bce_loss, and forward_1layer."""

from numpy.testing import assert_allclose

# ---- sigmoid tests ----
z = np.array([-2.0, 0.0, 0.5, 2.0])
assert_allclose(
    sigmoid(z),
    np.array([0.11920292202211755, 0.5, 0.6224593312018546, 0.8807970779778823]),
    rtol=1e-7,
    atol=1e-12,
)

# ---- bce_loss tests ----
y = np.array([0.0, 1.0])
y_hat = np.array([0.1, 0.9])
assert_allclose(bce_loss(y, y_hat, eps=1e-8), 0.10536050454671517, rtol=1e-7, atol=1e-12)


# also sanity-check it doesn't blow up on valid probabilities
y = np.array([0.0, 1.0])
y_hat = np.array([0.01, 0.99])
val = bce_loss(y, y_hat)
assert np.isfinite(val).all()

# ---- forward_1layer tests ----
X = np.array([
    [0.0, 0.0],
    [1.0, 1.0],
    [0.5, -1.0],
])
w = np.array([1.0, 2.0])
b = -0.5
y_pred = forward_1layer(X, w, b)
assert y_pred.shape == (X.shape[0],)
assert_allclose(
    y_pred,
    np.array([0.3775406687981454, 0.9241418199787566, 0.11920292202211755]),
    rtol=1e-7,
    atol=1e-12,
)

print("Success!")

<div class="alert alert-success">

## Exercise 2 (C–D)

**C.** Implement `train_1layer(X, y, lr, n_epochs)` using the update rules above. Return the final weights and the loss at every epoch.

This is **full-batch (batch) gradient descent**: instead of updating after each individual example (stochastic GD), you process *all* $n$ examples in a single forward pass, then update weights once per epoch. The structure looks like:

```
for each epoch:
    y_hat  ← forward_1layer(...)            # predictions for all n examples at once, shape (n,)
    loss   ← bce_loss(...)                  # record loss for this epoch
    delta  ← (y_hat − y) / n               # per-example error, already scaled by 1/n
    w     -= lr * (X.T @ delta)             # gradient w.r.t. w — see note below
    b     -= lr * sum(delta)
```

**Note on `X.T @ delta`:** The update formula $\frac{1}{n}\sum_{i=1}^n \delta_i \cdot x_{i,k}$ loops over all examples for each weight $k$. Writing it as `X.T @ delta` does exactly the same thing in one matrix multiply: row $k$ of $X^\top$ is all the $x_{i,k}$ values, so dotting with $\delta$ gives the sum. No loop over examples needed.

**D.** Train the network for 10 000 epochs with learning rate 0.5. Plot the training loss and calulate the average accuracy. Does it converge?

</div>

In [ ]:
def train_1layer(X, y, lr=0.5, n_epochs=10_000, seed=0):
    """
    Train single-layer network with full-batch gradient descent.

    Returns
    -------
    w : (2,)  — final weights
    b : float — final bias
    losses : (n_epochs,) — BCE loss at each epoch
    """
    rng = np.random.default_rng(seed)
    w = rng.normal(0, 0.1, 2)
    b = 0.0
    n = len(y)
    # Store the loss at each epoch
    losses = np.zeros(n_epochs)

    for epoch in range(n_epochs):
        # YOUR CODE HERE (REPLACE pass)
        pass

    return w, b, losses

In [ ]:
# the network is trained with 10 000 epochs with learning rate 0.5. by default.
w1, b1_1layer, losses_1layer = train_1layer(X, y)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(losses_1layer, 'steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.set_title('Single-layer network: training loss')
plt.tight_layout()
plt.show()

y_hat_1 = forward_1layer(X, w1, b1_1layer)
# YOUR CODE HERE: calculate accuracy with np.mean
# Hint 1: Convert probabilities to predicted class labels with a threshold at 0.5.
acc_1 = None
print(f'Single-layer accuracy: {acc_1:.3f}  (final loss: {losses_1layer[-1]:.4f})')

---

# 3. Why Does the Single Layer Fail?

<div class="alert alert-success">

## Exercise 3A

**A.** Plot the **decision boundary** of the trained single-layer network on top of the data.

   *Hint:* The boundary is where $\hat{y} = 0.5$, i.e. $\mathbf{w}^\top \mathbf{x} + b = 0$. Either draw this line analytically or use a meshgrid.

</div>

In [ ]:
def plot_decision_boundary(predict_fn, X, y, title, ax=None):
    """Plot a 2D decision boundary using a meshgrid."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300),
                          np.linspace(-0.5, 1.5, 300))
    Z = predict_fn(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[0, 0.5, 1],
                colors=['#AED6F1', '#F1948A'], alpha=0.5)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
    ax.scatter(X[y==0, 0], X[y==0, 1], c='steelblue', edgecolors='k', s=30, zorder=3)
    ax.scatter(X[y==1, 0], X[y==1, 1], c='tomato',    edgecolors='k', s=30, zorder=3)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(title)
    return ax


# Exercise 3A: Plot single-layer decision boundary
pred_fn = None # YOUR CODE HERE REPLACE None with the correct function

plot_decision_boundary(
    pred_fn,
    X, y,
    f'Single-layer  (accuracy = {acc_1:.0%})'
)
plt.tight_layout()
plt.show()


<div class="alert alert-success">

## Exercise 3B

In 2–3 sentences, explain geometrically why a single layer can never solve XOR, no matter how long you train it or what learning rate you use. Write down your answer below.

</div>


---

# 4. A Two-Layer Network
![Two layer visualization](two_layers.png)
Adding a **hidden layer** lets the network first transform the inputs into a new representation, and then classify in that new space. The network computes in two stages:

**Hidden layer** — $H$ units, each computing its own weighted sum of the inputs and then applying sigmoid:
$$z^{(1)}_j = w^{(1)}_{j,1} \cdot x_1 + w^{(1)}_{j,2} \cdot x_2 + b^{(1)}_j \qquad \text{for } j = 1, \ldots, H$$
$$h_j = \sigma(z^{(1)}_j)$$
Each hidden unit $h_j$ detects a different linear pattern in the input. Think of it as drawing a line in the input space and asking "which side of this line is the point on?"

**Output layer** — one unit, taking a weighted sum of the hidden activations:
$$z^{(2)} = w^{(2)}_1 \cdot h_1 + w^{(2)}_2 \cdot h_2 + b^{(2)}$$
$$\hat{y} = \sigma(z^{(2)})$$
The output unit combines the detections from the hidden layer into a final prediction.

**Parameters** (what the network learns):
- $w^{(1)}_{j,k}$: weight from input $k$ to hidden unit $j$ — stored as `W1[j, k]`
- $b^{(1)}_j$: bias of hidden unit $j$ — stored as `b1[j]`
- $w^{(2)}_j$: weight from hidden unit $j$ to the output — stored as `W2[j]`
- $b^{(2)}$: output bias — stored as `b2`

We use $H = 2$: the minimum number of hidden units needed to solve XOR.



<div class="alert alert-success">

## Exercise 4

**A.** Implement `forward_2layer(X, W1, b1, W2, b2)`. Return **all intermediate values** $(z^{(1)}, h, z^{(2)}, \hat{y})$ — you will need them in backpropagation.

**B.** Sanity-check: with all-zero weights, every output should be exactly 0.5. Verify this.

</div>

In [ ]:
def forward_2layer(X, W1, b1, W2, b2):
    """
    Two-layer forward pass.

    Parameters
    ----------
    X  : (n, 2)
    W1 : (H, 2)   hidden-layer weights
    b1 : (H,)     hidden-layer biases
    W2 : (H,)     output-layer weights
    b2 : float    output bias

    Returns
    -------
    z1    : (n, H)  pre-activation hidden
    h     : (n, H)  post-activation hidden
    z2    : (n,)    pre-activation output
    y_hat : (n,)    predicted probabilities
    """
    # YOUR CODE HERE
    z1    = 'REPLACE_ME'          # (n, H)
    h     = 'REPLACE_ME'              # (n, H)
    z2    = 'REPLACE_ME'             # (n,)
    y_hat = 'REPLACE_ME'             # (n,)
    return z1, h, z2, y_hat

In [ ]:
"""Check forward_2layer intermediate values (numeric expected)."""

from numpy.testing import assert_allclose

# Test inputs
X = np.array([
    [0.0, 1.0],
    [1.0, 0.0],
    [0.5, 0.5],
])  # (n,2) with n=3

# Hidden layer size H=2
W1 = np.array([
    [1.0, -1.0],   # for hidden unit j=0
    [0.5,  0.25],  # for hidden unit j=1
])  # (H,2)

b1 = np.array([0.1, -0.2])     # (H,)
W2 = np.array([1.5, -1.0])     # (H,)
b2 = 0.3                        # scalar

# Run
z1, h, z2, y_hat = forward_2layer(X, W1, b1, W2, b2)

# Shape checks
assert z1.shape == (3, 2)
assert h.shape == (3, 2)
assert z2.shape == (3,)
assert y_hat.shape == (3,)

# Numeric expected values
expected_z1 = np.array([
    [-0.9,   0.05 ],
    [ 1.1,   0.3  ],
    [ 0.1,   0.175],
])

expected_h = np.array([
    [0.289050497374996, 0.5124973964842103],
    [0.7502601055951177, 0.574442516811659],
    [0.52497918747894,  0.5436386872370789],
])

expected_z2 = np.array([
    0.2210783495782837,
    0.8509476415810175,
    0.5438300939813312,
])

expected_y_hat = np.array([
    0.5550455708699733,
    0.7007658941313016,
    0.6327029445373548,
])

assert_allclose(z1, expected_z1, rtol=1e-7, atol=1e-12)
assert_allclose(h, expected_h, rtol=1e-7, atol=1e-12)
assert_allclose(z2, expected_z2, rtol=1e-7, atol=1e-12)
assert_allclose(y_hat, expected_y_hat, rtol=1e-7, atol=1e-12)

print("Success!")

In [ ]:
# Exercise 4B: Sanity check with all-zero weights
H = 2
W1_test = np.zeros((H, 2))
b1_test = np.zeros(H)
W2_test = np.zeros(H)
b2_test = 0.0

_, _, _, y_hat_test = forward_2layer(X[:4], W1_test, b1_test, W2_test, b2_test)
print('With all-zero weights, outputs should all be 0.5:')
print(y_hat_test)

---

# 5. Backpropagation

We need to know how much each weight contributed to the error so we can update it. The key idea is the **chain rule**: if weight $w$ affects the output through several intermediate steps, we multiply the sensitivities of each step together. We work backwards from the output — hence *backpropagation*. With a binary cross-entropy loss and sigmoid activation function, the backpropagation is even a bit simpler than we saw in class. Here are the steps

**Step 1 — Output error**

For each training example $i$, compute the scaled output error:
$$\delta^{(2)}_i = \frac{\hat{y}_i - y_i}{n}$$

**Step 2 — Update the output weights**

Each output weight $w^{(2)}_j$ contributed to the error in proportion to how active hidden unit $j$ was:
$$w^{(2)}_j \leftarrow w^{(2)}_j - \eta \sum_{i=1}^n \delta^{(2)}_i \cdot h_{i,j}$$
$$b^{(2)} \leftarrow b^{(2)} - \eta \sum_{i=1}^n \delta^{(2)}_i$$
(Same logic as the single-layer update, but using the hidden activations $h$ instead of the raw inputs.)

**Step 3 — Pass the error back to the hidden layer**

Each hidden unit $j$ "receives" a share of the output error, weighted by its output weight $w^{(2)}_j$. We also multiply by how sensitive the hidden sigmoid was at that point:
$$\delta^{(1)}_{i,j} = \delta^{(2)}_i \cdot w^{(2)}_j \cdot \sigma'(z^{(1)}_{i,j})$$
where the **sigmoid derivative** is $\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$.

A hidden unit that was in the "flat" part of the sigmoid (very high or very low activation) gets a small error signal because changing its weights would barely affect the output.

**Step 4 — Update the hidden weights**

Now use the hidden errors $\delta^{(1)}_{i,j}$ exactly as you used $\delta^{(2)}_i$ in Step 2, but with the original inputs $x_{i,k}$ instead of the hidden activations:
$$w^{(1)}_{j,k} \leftarrow w^{(1)}_{j,k} - \eta \sum_{i=1}^n \delta^{(1)}_{i,j} \cdot x_{i,k}$$
$$b^{(1)}_j \leftarrow b^{(1)}_j - \eta \sum_{i=1}^n \delta^{(1)}_{i,j}$$

<div class="alert alert-success">

## Exercise 5

**A.** Implement `backprop(X, y, z1, h, y_hat, W2)` following the four steps above. It should return the weight updates `(dW1, db1, dW2, db2)`.

**B.** Implement `train_2layer(X, y, n_hidden, lr, n_epochs)` that initialises the weights randomly and runs gradient descent using your `forward_2layer` and `backprop` functions.

**C.** Train the two-layer network ($H=2$, lr = 0.5, 10 000 epochs). Plot the training loss alongside the single-layer loss from Exercise 2.

</div>

In [ ]:
def sigmoid_deriv(z):
    """Derivative of sigmoid: sigma(z) * (1 - sigma(z))."""
    s = sigmoid(z)
    return s * (1 - s)


def backprop(X, y, z1, h, y_hat, W2):
    """
    Compute gradients via backpropagation.

    Parameters
    ----------
    X     : (n, 2)
    y     : (n,)    true labels
    z1    : (n, H)  pre-activation hidden (from forward pass)
    h     : (n, H)  post-activation hidden (from forward pass)
    y_hat : (n,)    predictions (from forward pass)
    W2    : (H,)    current output weights (needed for delta1)

    Returns
    -------
    dW1 : (H, 2)
    db1 : (H,)
    dW2 : (H,)
    db2 : float
    """
    n = len(y)

    # YOUR CODE HERE
    # Output-layer error
    delta2 = 'REPLACE_ME'                         # (n,)
    dW2    = 'REPLACE_ME'                              # (H,)
    db2    = 'REPLACE_ME'                            # scalar

    # Hidden-layer error (broadcast delta2 over H, then multiply by sigma')
    delta1 = delta2[:, None] * W2[None, :] * sigmoid_deriv(z1)  # (n, H)
    dW1    = 'REPLACE_ME'                              # (H, 2)
    db1    = 'REPLACE_ME'                    # (H,)

    return dW1, db1, dW2, db2


def train_2layer(X, y, n_hidden=2, lr=0.5, n_epochs=10_000, seed=42):
    """
    Train two-layer network with full-batch gradient descent.

    Returns
    -------
    W1, b1, W2, b2 : final parameters
    losses         : (n_epochs,) BCE loss at each epoch
    """
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 0.5, (n_hidden, 2))
    b1 = np.zeros(n_hidden)
    W2 = rng.normal(0, 0.5, n_hidden)
    b2 = 0.0
    losses = np.zeros(n_epochs)

    # YOUR CODE HERE (REPLACE pass)
    for epoch in range(n_epochs):
        pass

    return W1, b1, W2, b2, losses

In [ ]:
"""Check backprop gradients with numeric expected values."""

from numpy.testing import assert_allclose

# Small toy problem
X = np.array([
    [0.0, 1.0],
    [1.0, 0.0],
    [0.5, 0.5],
])  # (n,2), n=3

# Hidden size H=2
W1 = np.array([
    [1.0, -1.0],
    [0.5,  0.25],
])  # (H,2)

b1 = np.array([0.1, -0.2])   # (H,)
W2 = np.array([1.5, -1.0])    # (H,)
b2 = 0.3

# Labels y (shape (n,))
y = np.array([0.0, 1.0, 1.0])

# Get intermediates from forward pass (matches backprop signature)
z1, h, z2, y_hat = forward_2layer(X, W1, b1, W2, b2)

# Run backprop
dW1, db1, dW2, db2 = backprop(X, y, z1, h, y_hat, W2)

# Shape checks
assert dW1.shape == (2, 2)   # (H,2)
assert db1.shape == (2,)     # (H,)
assert dW2.shape == (2,)     # (H,)
assert np.isscalar(db2) or np.shape(db2) == ()  # scalar

# Numeric expected gradients
expected_dW1 = np.array([
    [-0.05093250050300988,  0.034132246384924916],
    [ 0.039570889310119664, -0.03103743312020775 ],
])

expected_db1 = np.array([
    -0.01680025411808497,
     0.008533456189911914,
])

expected_dW2 = np.array([
    -0.08563017442704753,
    -0.02903675731596831,
])

expected_db2 = -0.03716186348712343

assert_allclose(dW1, expected_dW1, rtol=1e-6, atol=1e-12)
assert_allclose(db1, expected_db1, rtol=1e-6, atol=1e-12)
assert_allclose(dW2, expected_dW2, rtol=1e-6, atol=1e-12)
assert_allclose(db2, expected_db2, rtol=1e-6, atol=1e-12)

print("Success!")

In [ ]:
# Exercise 5C: Train and compare losses

W1, b1, W2, b2, losses_2layer = train_2layer(X, y)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_1layer, 'steelblue', label='Single-layer')
ax.plot(losses_2layer, 'tomato',    label='Two-layer')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.set_title('Training loss: single-layer vs two-layer')
ax.legend()
plt.tight_layout()
plt.show()

_, _, _, y_hat_2 = forward_2layer(X, W1, b1, W2, b2)
# YOUR CODE HERE: calculate accuracy with np.mean
# Hint 1: Convert probabilities to predicted class labels with a threshold at 0.5.
acc_2 = None
print(f'Single-layer accuracy: {acc_1:.3f}')
print(f'Two-layer   accuracy: {acc_2:.3f}')

---

# 6. Visualising the Solution

<div class="alert alert-success">

## Exercise 6A

**A.** Plot the two-layer network's decision boundary side-by-side with the single-layer boundary (reuse `plot_decision_boundary`).

</div>

In [ ]:
# Exercise 6A: Side-by-side decision boundaries

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
# YOUR CODE HERE
pred_fn = None # YOUR CODE HERE REPLACE None with the correct function
plot_decision_boundary(
    pred_fn,
    X, y, f'Single-layer  (acc = {acc_1:.0%})', ax=axes[0]
)

pred_fn_2layer = None # YOUR CODE HERE REPLACE None with the correct function
plot_decision_boundary(
    pred_fn_2layer,
    X, y, f'Two-layer  (acc = {acc_2:.0%})', ax=axes[1]
)
plt.tight_layout()
plt.show()

---

<div class="alert alert-success">

## Exercise 6B

**B.** Visualise **what the hidden units learned**: for each hidden unit $i$, plot the linear boundary $\mathbf{W}_1[i,:] \cdot \mathbf{x} + b_1[i] = 0$ (i.e. where its pre-activation is zero) on top of the data. Label each line with the output weight $W_2[i]$ it was assigned.


</div>

In [ ]:
# Exercise 6B: Hidden unit boundaries

fig, ax = plt.subplots(figsize=(5, 5))

# YOUR CODE HERE add the scatter plot for X. 
ax.scatter('REPLACE_ME', 'REPLACE_ME', c='steelblue', edgecolors='k', s=30, alpha=0.5, zorder=3)
ax.scatter('REPLACE_ME', 'REPLACE_ME', c='tomato',    edgecolors='k', s=30, alpha=0.5, zorder=3)

x_range = np.array([-0.5, 1.5])
colors   = ['purple', 'darkorange']

for i in range(len(b1)):
    w_x, w_y = W1[i, 0], W1[i, 1]
    # Boundary: w_x * x1 + w_y * x2 + b1[i] = 0  =>  x2 = -(w_x * x1 + b1[i]) / w_y
    if abs(w_y) > 1e-6:
        y_line = -(w_x * x_range + b1[i]) / w_y
        ax.plot(x_range, y_line, '--', color=colors[i], linewidth=2.5,
                label=f'Hidden unit {i+1}  ($W_2$={W2[i]:+.2f})')

ax.set_xlim([-0.5, 1.5]); ax.set_ylim([-0.5, 1.5])
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Boundaries learned by each hidden unit')
ax.legend()
plt.tight_layout()
plt.show()

print('Hidden layer weights W1:')
for i in range(len(b1)):
    print(f'  Unit {i+1}: W1={W1[i]}, b1={b1[i]:.3f}, output weight W2={W2[i]:.3f}')

---

<div class="alert alert-success">

## Exercise 6C

**C.** In 2–3 sentences, explain what the two hidden units are computing and how combining them solves XOR.
Write down your answer below.
</div>

In [ ]:
## Restart the kernel and rerun the whole notebook to verify it runs without errors
print('No errors!')